In [1]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
if os.environ.get("GEMINI_API_KEY"):
    print("API key exists")
else:
    raise ValueError("GEMINI_API_KEY not found")


API key exists


DOC LOADER


In [2]:
from langchain_community.document_loaders import PyPDFLoader
import pypdf
loader = PyPDFLoader("A Game Of Thrones - George RR Martin.pdf")
pages = loader.load()
for p in pages[:20]:
    print(p.metadata["page"], len(p.page_content), repr(p.page_content[:80]))

print(pages[11].metadata)
print("____________________________")

import pprint
pprint.pprint(pages[11].metadata)




C:\Users\Benjamin\AppData\Local\Temp\ipykernel_2500\3733184858.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


0 0 ''
1 0 ''
2 1095 'A Game of Thrones\nA Bantam Spectra Book\nSPECTRA and the portrayal of a boxed “s”'
3 23 'this one is for Melinda'
4 227 'CONTENTS\nCOVER\nTITLE PAGE\nCOPYRIGHT\nDEDICATION\nMAPS\nPROLOGUE\nBRAN\nCATELYN\nDAENER'
5 240 'EDDARD\nCATELYN\nSANSA\nEDDARD\nTYRION\nARYA\nEDDARD\nCATELYN\nEDDARD\nDAENERYS\nBRAN\nTYRI'
6 235 'CATELYN\nDAENERYS\nARYA\nBRAN\nSANSA\nDAENERYS\nTYRION\nJON\nCATELYN\nDAENERYS\nAPPENDIX\nH'
7 0 ''
8 0 ''
9 0 ''
10 1913 'PROLOGUE\n“We should start back,” Gared urged as the woods began to grow dark\naro'
11 2372 'Until tonight. Something was different tonight. There was an edge to this\ndarkne'
12 2101 'Will had been a hunter before he joined the Night’s Watch. Well, a\npoacher in tr'
13 2330 'and at first you shiver and your teeth chatter and you stamp your feet and\ndream'
14 2045 'the careless and the unwary. Ser Waymar Royce came next, his great black\ndestrie'
15 2192 'Gared dismounted. “We need a fire. I’ll see to it.”\n“How big a fool are y

In [3]:
POV_HEADINGS = {"PROLOGUE", "BRAN", "CATELYN", "DAENERYS", "EDDARD",
                "SANSA", "TYRION", "ARYA", "JON", "APPENDIX"}

FIRST_CONTENT_PAGE = 10
pages = [p for p in pages if p.metadata["page"] >= FIRST_CONTENT_PAGE]

chapter_no = 0
current_chapter = "Unknown"

for doc in pages:
    first_line = doc.page_content.strip().split("\n")[0].strip()

    if first_line in POV_HEADINGS:
        if first_line == "PROLOGUE":
            current_chapter = "Prologue"
        elif first_line == "APPENDIX":
            current_chapter = "Appendix"
        else:
            chapter_no += 1
            current_chapter = f"Chapter {chapter_no}: {first_line.title()}"

    doc.metadata["chapter"] = current_chapter

VERIFICATION

In [4]:
print("pages kept:", len(pages), "| chapters:", chapter_no)   # expect 72
print("Unknown pages:", [p.metadata["page"] for p in pages if p.metadata["chapter"] == "Unknown"])  # expect []

appendix = [p for p in pages if p.metadata["chapter"] == "Appendix"]
print("appendix pages:", len(appendix))
print(repr(pages[-1].page_content[:200]))

pages kept: 776 | chapters: 72
Unknown pages: []
appendix pages: 20
'ACKNOWLEDGMENTS\nThe devil is in the details, they say.\nA book this size has a lot of devils, any one of which will bite you if you\ndon’t watch out. Fortunately, I know a lot of angels.\nThanks and appr'


In [5]:
#trimming the metadata
for doc in pages:
    doc.metadata = {
        "source": "A Game of Thrones",
        "page": doc.metadata["page"] + 1,
        "chapter": doc.metadata["chapter"],
    }

In [6]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(pages)

print(len(chunks))

2136


In [7]:
print(chunks[0].metadata)
print(chunks[500].page_content[:])

{'source': 'A Game of Thrones', 'page': 11, 'chapter': 'Prologue'}
softly. “Lord Arryn was a prudent man, but I fear that His Grace does not
always listen to wise counsel.”
“My royal brother loves tournaments and feasts,” Renly Baratheon said,
“and he loathes what he calls ‘counting coppers.’”
“I will speak with His Grace,” Ned said. “This tourney is an
extravagance the realm cannot afford.”
“Speak to him as you will,” Lord Renly said, “we had still best make our
plans.”
“Another day,” Ned said. Perhaps too sharply, from the looks they gave
him. He would have to remember that he was no longer in Winterfell, where
only the king stood higher; here, he was but first among equals. “Forgive
me, my lords,” he said in a softer tone. “I am tired. Let us call a halt for
today and resume when we are fresher.” He did not ask for their consent,
but stood abruptly, nodded at them all, and made for the door.
Outside, wagons and riders were still pouring through the castle gates,
and the yard was a c

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    collection_name="got_local",
    embedding_function=embeddings,
    persist_directory="./chroma_db_local",
)
if vectorstore._collection.count() == 0:
    vectorstore.add_documents(chunks)
print(vectorstore._collection.count())   # expect 2136




d:\Experiments\ONE-BOOK\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1388.67it/s]


2136


In [9]:
results = vectorstore.similarity_search("how is the relationship between Eddard Stark and Robert Baratheon?", k=4)
for r in results:
    print(r.metadata["chapter"], "| p.", r.metadata["page"])
    print(r.page_content[:200].replace("\n", " "))
    print("---")

Chapter 49: Eddard | p. 503
you would be so kind?” The Lord Commander of the Kingsguard examined the paper. “King Robert’s seal, and unbroken.” He opened the letter and read. “Lord Eddard Stark is herein named Protector of the R
---
Chapter 4: Eddard | p. 52
“I understand,” Ned said softly. Robert looked at him. “I think you do. If so, you are the only one, my old friend.” He smiled. “Lord Eddard Stark, I would name you the Hand of the King.” Ned dropped 
---
Chapter 30: Eddard | p. 295
EDDARD “I stood last vigil for him myself,” Ser Barristan Selmy said as they looked down at the body in the back of the cart. “He had no one else. A mother in the Vale, I am told.” In the pale dawn li
---
Chapter 4: Eddard | p. 45
EDDARD The visitors poured through the castle gates in a river of gold and silver and polished steel, three hundred strong, a pride of bannermen and knights, of sworn swords and freeriders. Over their
---


In [10]:
def show(query, k=8):
    print("Q:", query)
    for r in vectorstore.similarity_search(query, k=k):
        print(f"  [{r.metadata['chapter']} | p.{r.metadata['page']}]",
              r.page_content[:350].replace("\n", " "))
    print()

show("Why does Bran fall from the tower?")



Q: Why does Bran fall from the tower?
  [Chapter 24: Bran | p.239] wait and see.” After a time he slept. In his dream he was climbing again, pulling himself up an ancient windowless tower, his fingers forcing themselves between blackened stones, his feet scrabbling for purchase. Higher and higher he climbed, through the clouds and into the night sky, and still the tower rose before him. When he paused to look down
  [Chapter 17: Bran | p.159] BRAN It seemed as though he had been falling for years. Fly, a voice whispered in the darkness, but Bran did not know how to fly, so all he could do was fall. Maester Luwin made a little boy of clay, baked him till he was hard and brittle, dressed him in Bran’s clothes, and flung him off a roof. Bran remembered the way he shattered. “But I never fa
  [Chapter 8: Bran | p.85] father had been born, a lightning strike had set it afire. The top third of the structure had collapsed inward, and the tower had never been rebuilt. Sometimes his father sent

In [11]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.2
)



In [12]:
import re
from rank_bm25 import BM25Okapi

tok = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tok(c.page_content) for c in chunks])

def keyword_search(query, k=5):
    scores = bm25.get_scores(tok(query))
    top = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)[:k]
    return [chunks[i] for i in top]

def hybrid_search(query, k=5):
    seen, out = set(), []
    for d in vectorstore.similarity_search(query, k=k) + keyword_search(query, k=k):
        key = (d.metadata["page"], d.page_content[:60])
        if key not in seen:
            seen.add(key)
            out.append(d)
    return out

print([d.metadata["page"] for d in keyword_search("Theon hostage ward rebellion", k=10)])

[780, 46, 383, 134, 45, 57, 28, 770, 389, 22]


In [13]:
def expand_queries(question, n=3):
    prompt = f"""You help search the novel A Game of Thrones.
For the question below, write:
- {n} alternative search queries, one per line, each starting with "Q: ".
  Use names and events the book itself would use.
- one line starting with "K: " with 6-10 single keywords likely to appear in the
  passage that answers it, including synonyms and background events
  (example: for a question about why someone is hated, K: grudge, betrayal, insult, feud, revenge).

Question: {question}"""
    text = llm.invoke(prompt).content[0]["text"]
    lines = text.splitlines()
    queries = [l[2:].strip() for l in lines if l.startswith("Q:")][:n]
    keywords = " ".join(l[2:].strip() for l in lines if l.startswith("K:"))
    return [question] + queries, keywords

def retrieve(question, k=3):
    queries, keywords = expand_queries(question)
    seen, hits = set(), []

    def add(docs):
        for d in docs:
            key = (d.metadata["page"], d.page_content[:60])
            if key not in seen:
                seen.add(key)
                hits.append(d)

    for q in queries:
        add(hybrid_search(q, k=k))
    if keywords:
        add(keyword_search(keywords, k=6))   # one BM25 search on the rare words
    return hits

In [14]:

from collections import defaultdict

# 1. build lookups once (re-run if you ever re-chunk)
by_page = defaultdict(list)
for c in chunks:
    by_page[c.metadata["page"]].append(c)

order = {id(c): i for i, c in enumerate(chunks)}   # keeps reading order

# 2. add the following pages for every hit
def with_neighbors(docs, after=2):
    seen, out = set(), []
    for d in docs:
        p0, ch = d.metadata["page"], d.metadata["chapter"]
        for p in range(p0, p0 + after + 1):
            for c in by_page.get(p, []):
                if c.metadata["chapter"] == ch:
                    key = (p, c.page_content[:60])
                    if key not in seen:
                        seen.add(key)
                        out.append(c)
    return sorted(out, key=lambda c: order[id(c)])

# 3. use it in ask()
def ask(question, k=3, after=1):
    hits = retrieve(question, k=k)
    docs = with_neighbors(hits, after=after)
    print("hits:", len(hits), "-> chunks sent to the model:", len(docs))

    context = "\n\n".join(
        f"[{d.metadata['chapter']}, p.{d.metadata['page']}]\n{d.page_content}"
        for d in docs
    )
    prompt = f"""Answer the question using only the excerpts below from A Game of Thrones.
If the excerpts don't contain the answer, say so instead of guessing.
Lead with a direct answer in one or two sentences, then add key supporting details
with chapter and page. Don't list every minor detail.


Excerpts:
{context}

Question: {question}"""

    answer = llm.invoke(prompt).content[0]["text"]
    sources = sorted({(d.metadata["chapter"], d.metadata["page"]) for d in docs})
    return answer, sources

answer, sources = ask("Why does Bran fall from the tower?")
print(answer)
print(sources)

d:\Experiments\ONE-BOOK\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


hits: 20 -> chunks sent to the model: 76


d:\Experiments\ONE-BOOK\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Bran falls from the tower because he is startled while peering in a window and loses his grip after the queen pushes her partner away and screams upon seeing him (Chapter 8, pp. 88–89). 

Key supporting details:
* Bran climbs up and hangs upside down from a gargoyle to look into a window of the First Keep where a man and a woman are talking and wrestling (Chapter 8, pp. 88–89). 
* When he makes a noise, the woman (identified as the queen) opens her eyes, sees him staring at her, and screams in shock (Chapter 8, p. 89). 
* As the woman wildly pushes the man away and points, Bran panics and hurries to pull himself up, causing his hands and feet to slip off the stone and sending him falling (Chapter 8, p. 89).
[('Chapter 14: Catelyn', 131), ('Chapter 14: Catelyn', 132), ('Chapter 17: Bran', 159), ('Chapter 17: Bran', 160), ('Chapter 17: Bran', 161), ('Chapter 17: Bran', 162), ('Chapter 22: Arya', 214), ('Chapter 22: Arya', 215), ('Chapter 24: Bran', 233), ('Chapter 24: Bran', 234), ('Chap

TEST

In [15]:
tests = [
    "Who is Gared?",
    "What happens to Ser Waymar Royce?",
    "Who is Jon Snow's mother?",                 # the book doesn't answer this
    "What is Tyrion's favorite color?",          # not in the book at all
    "What happens in the Prologue?",             # a broad question
    "Why is Theon Greyjoy at winterfell?",
    "Who is Theon Grejoy?"
]
for q in tests:
    print("Q:", q)
    a, s = ask(q)
    print(a, "\n", s, "\n" + "-" * 60)

Q: Who is Gared?


d:\Experiments\ONE-BOOK\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


hits: 18 -> chunks sent to the model: 69


d:\Experiments\ONE-BOOK\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Gared is an old man-at-arms in the Night's Watch who has spent forty years there, man and boy, and later deserts the order. 

Supporting details:
* **Prologue, p. 11**: He is described as "an old man, past fifty" who has spent "forty years in the Night's Watch, man and boy."
* **Chapter 21: Tyrion, p. 202**: Lord Commander Mormont notes that Gared was "near as old as I am and longer on the Wall," but forswore himself and fled, leading Lord Eddard to send his head to Winterfell. 
 [('Chapter 19: Jon', 176), ('Chapter 19: Jon', 177), ('Chapter 19: Jon', 181), ('Chapter 19: Jon', 182), ('Chapter 19: Jon', 183), ('Chapter 21: Tyrion', 202), ('Chapter 21: Tyrion', 203), ('Chapter 21: Tyrion', 208), ('Chapter 21: Tyrion', 209), ('Chapter 41: Jon', 428), ('Chapter 41: Jon', 429), ('Chapter 48: Jon', 499), ('Chapter 48: Jon', 500), ('Chapter 70: Jon', 737), ('Chapter 70: Jon', 738), ('Chapter 70: Jon', 741), ('Chapter 70: Jon', 742), ('Prologue', 11), ('Prologue', 12), ('Prologue', 13), ('Prol

...

Theon Greyjoy is at Winterfell as a hostage and ward of Lord Eddard Stark ([Chapter 4: Eddard, p.46]). 

Lord Stark took Theon as a hostage following Balon Greyjoy’s rebellion after King Robert accepted the rebel lord's surrender at Greyjoy's stronghold ([Chapter 4: Eddard, p.45-46]). 
 [('Appendix', 770), ('Appendix', 771), ('Appendix', 780), ('Appendix', 781), ('Chapter 12: Eddard', 119), ('Chapter 13: Tyrion', 120), ('Chapter 13: Tyrion', 121), ('Chapter 14: Catelyn', 134), ('Chapter 14: Catelyn', 135), ('Chapter 14: Catelyn', 136), ('Chapter 14: Catelyn', 137), ('Chapter 1: Bran', 22), ('Chapter 1: Bran', 23), ('Chapter 24: Bran', 232), ('Chapter 24: Bran', 233), ('Chapter 30: Eddard', 298), ('Chapter 30: Eddard', 299), ('Chapter 30: Eddard', 303), ('Chapter 30: Eddard', 304), ('Chapter 31: Tyrion', 325), ('Chapter 31: Tyrion', 326), ('Chapter 37: Bran', 385), ('Chapter 37: Bran', 386), ('Chapter 45: Eddard', 464), ('Chapter 45: Eddard', 465), ('Chapter 4: Eddard', 45), ('Chapter 4

d:\Experiments\ONE-BOOK\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


hits: 21 -> chunks sent to the model: 96


d:\Experiments\ONE-BOOK\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


The excerpts state that Theon Greyjoy is Lord Eddard Stark's ward and the heir to the Iron Islands. 

Key supporting details:
* In the appendix (p. 770), he is listed among Lord Eddard's household and family as "his ward, THEON GREYJOY, heir to the Iron Islands."
* He is further identified as the son of Balon Greyjoy and the sole surviving son and heir to Pyke (Appendix, p. 780-781). 
 [('Appendix', 770), ('Appendix', 771), ('Appendix', 779), ('Appendix', 780), ('Appendix', 781), ('Chapter 14: Catelyn', 134), ('Chapter 14: Catelyn', 135), ('Chapter 14: Catelyn', 136), ('Chapter 1: Bran', 22), ('Chapter 1: Bran', 23), ('Chapter 20: Eddard', 187), ('Chapter 20: Eddard', 188), ('Chapter 24: Bran', 232), ('Chapter 24: Bran', 233), ('Chapter 30: Eddard', 303), ('Chapter 30: Eddard', 304), ('Chapter 37: Bran', 383), ('Chapter 37: Bran', 384), ('Chapter 37: Bran', 385), ('Chapter 37: Bran', 386), ('Chapter 37: Bran', 389), ('Chapter 37: Bran', 390), ('Chapter 47: Eddard', 480), ('Chapter 47: 

In [16]:
def summarize_chapter(chapter_name):
    docs = [c for c in chunks if c.metadata["chapter"] == chapter_name]  # already in book order
    context = "\n\n".join(f"[p.{d.metadata['page']}]\n{d.page_content}" for d in docs)
    prompt = f"""Summarize what happens in this chapter of A Game of Thrones, in order.
Mention page numbers for the main events.

{context}"""
    return llm.invoke(prompt).content[0]["text"]

print(summarize_chapter("Prologue"))


d:\Experiments\ONE-BOOK\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3914: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Here is an ordered summary of the events in the prologue of *A Game of Thrones*, along with the page numbers where the main events occur.

*   **[p. 11–12] The Ranging Beyond the Wall:** Three members of the Night’s Watch—Ser Waymar Royce, Gared, and Will—are tracking a band of wildling raiders deep in the haunted forest. Night is falling, and a tense, unnatural cold sets in. Gared urges a retreat, but the arrogant, inexperienced young commander, Ser Waymar Royce, dismisses his fears and insists on continuing.
*   **[p. 12–14] Discussing the Dead Wildlings:** Will recounts what he discovered earlier: eight dead wildlings lying motionless in a camp without a fire or any visible wounds. Gared insists they simply froze to death, arguing that the cold is the deadliest enemy north of the Wall. Royce is skeptical, pointing out that the Wall was "weeping" (melting) recently, meaning it wasn't cold enough to kill eight prepared men. Royce orders Will to lead them to the camp.
*   **[p. 15–16] 

In [17]:
import pickle
with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)